# **1. Perkenalan Dataset**

Dataset yang digunakan adalah **Breast Cancer Wisconsin (Diagnostic)** dari sklearn built-in datasets.

- **Sumber**: `sklearn.datasets.load_breast_cancer()` (UCI ML Repository)
- **Jumlah sampel**: 569 sampel
- **Jumlah fitur**: 30 fitur numerik (radius, texture, perimeter, area, dll.)
- **Target (label)**: Binary classification
  - `0` = malignant (ganas) — 212 sampel
  - `1` = benign (jinak) — 357 sampel

Dataset ini digunakan untuk mengklasifikasikan tumor payudara berdasarkan karakteristik sel yang diukur dari gambar biopsi digital.

# **2. Import Library**

Pada tahap ini, diimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan preprocessing.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from scipy import stats

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid')

print('Libraries imported successfully!')

# **3. Memuat Dataset**

Dataset dimuat langsung dari sklearn built-in datasets. Dataset asli disimpan ke file CSV sebagai raw dataset.

In [ ]:
# Load Breast Cancer dataset dari sklearn
bc = load_breast_cancer()

df = pd.DataFrame(bc.data, columns=bc.feature_names)
df['target'] = bc.target
df['target_name'] = df['target'].map({0: 'malignant', 1: 'benign'})

# Simpan dataset asli ke CSV
os.makedirs('dataset_asli', exist_ok=True)
df.to_csv('dataset_asli/breast_cancer.csv', index=False)

print(f'Dataset dimuat: {df.shape[0]} sampel, {df.shape[1]} kolom')
print(f'Target classes: {bc.target_names.tolist()}')
print(f'Distribusi target:')
print(df['target_name'].value_counts())
df.head()

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini dilakukan analisis eksploratif untuk memahami karakteristik dataset, termasuk statistik deskriptif, distribusi kelas, korelasi antar fitur, dan visualisasi outlier.

In [ ]:
# Statistik deskriptif
print('Statistik Deskriptif:')
df.describe().round(3)

In [ ]:
# Cek missing values
missing = df.isnull().sum()
print('Missing Values:')
if missing.sum() == 0:
    print('Tidak ada missing values!')
else:
    print(missing[missing > 0])

In [ ]:
# Distribusi kelas target
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['target_name'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#2196F3', '#4CAF50'])
axes[0].set_title('Distribusi Kelas Target')
axes[0].set_xlabel('Kelas')
axes[0].set_ylabel('Jumlah Sampel')
for i, (name, val) in enumerate(counts.items()):
    axes[0].text(i, val + 1, str(val), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#2196F3', '#4CAF50'])
axes[1].set_title('Proporsi Kelas Target')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Distribusi fitur utama per kelas
feature_subset = ['mean radius', 'mean texture', 'mean perimeter',
                  'mean area', 'mean smoothness', 'mean compactness']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(feature_subset):
    for cls, color in zip(['malignant', 'benign'], ['#2196F3', '#4CAF50']):
        data = df[df['target_name'] == cls][col]
        axes[i].hist(data, bins=20, alpha=0.6, color=color, label=cls)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].legend()

plt.suptitle('Distribusi Fitur Utama per Kelas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap korelasi
feature_cols = [c for c in df.columns if c not in ['target', 'target_name']]
corr = df[feature_cols[:10] + ['target']].corr()

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Heatmap Korelasi Fitur', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplot fitur per kelas
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(feature_subset):
    data_malignant = df[df['target_name'] == 'malignant'][col].values
    data_benign = df[df['target_name'] == 'benign'][col].values
    bp = axes[i].boxplot([data_malignant, data_benign], patch_artist=True,
                          labels=['malignant', 'benign'])
    for patch, color in zip(bp['boxes'], ['#2196F3', '#4CAF50']):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[i].set_title(col, fontsize=10, fontweight='bold')

plt.suptitle('Distribusi Fitur per Kelas (Boxplot)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplot_features.png', dpi=120, bbox_inches='tight')
plt.show()

# **5. Data Preprocessing**

Pada tahap ini dilakukan preprocessing untuk memastikan kualitas data. Tahapan yang dilakukan:
1. Menangani Missing Values (pengisian dengan nilai median)
2. Deteksi dan Penghapusan Outlier menggunakan Z-Score (threshold = 3.0)
3. Menyimpan dataset hasil preprocessing ke file CSV

In [ ]:
# 4.1 Handle missing values
print('Sebelum handling missing values:')
print(f'  Shape: {df.shape}')
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f'  Kolom {col}: diisi dengan median={median_val:.3f}')
if df.isnull().sum().sum() == 0:
    print('  Tidak ada missing values yang ditemukan!')
print(f'Shape setelah handling: {df.shape}')

In [ ]:
# 4.2 Hapus outlier menggunakan Z-Score (threshold=3.0)
feat_cols = [c for c in numeric_cols if c != 'target']
z_scores = np.abs(stats.zscore(df[feat_cols]))
outlier_mask = (z_scores < 3.0).all(axis=1)
df_preprocessed = df[outlier_mask].reset_index(drop=True)

print(f'Sampel sebelum: {len(df)}')
print(f'Sampel setelah outlier removal: {len(df_preprocessed)}')
print(f'Outlier dihapus: {len(df) - len(df_preprocessed)} sampel')
print(f'Distribusi kelas setelah preprocessing:')
print(df_preprocessed['target_name'].value_counts())

In [ ]:
# Simpan dataset hasil preprocessing ke CSV
os.makedirs('dataset_preprocessing', exist_ok=True)
df_preprocessed.to_csv(
    'dataset_preprocessing/breast_cancer_preprocessing.csv', index=False)

# Juga simpan di root
df_preprocessed.to_csv('dataset_preprocessing.csv', index=False)

print('Dataset hasil preprocessing berhasil disimpan!')
print(f'Shape: {df_preprocessed.shape}')
print('File: dataset_preprocessing/breast_cancer_preprocessing.csv')
print('File: dataset_preprocessing.csv')

In [ ]:
# Ringkasan akhir
n_orig = len(df)
n_clean = len(df_preprocessed)
print('=' * 50)
print('RINGKASAN PREPROCESSING')
print('=' * 50)
print(f'Dataset asli         : {n_orig} sampel, {df.shape[1]} kolom')
print(f'Setelah preprocessing: {n_clean} sampel, {df_preprocessed.shape[1]} kolom')
print(f'Outlier dihapus      : {n_orig - n_clean} sampel')
classes = df_preprocessed['target_name'].unique().tolist()
print(f'Target classes       : {classes}')
print('=' * 50)
df_preprocessed.head()